# Llama 3 8B — IB Quantization Pipeline

Same logic as GPT-2 Stage 2+3 but scaled to Llama 3 8B.

**Before running:**
- Close all other applications (browser tabs, Slack, etc.)
- You need a Hugging Face account with Llama 3 access approved
- Run `huggingface-cli login` in terminal first
- 16GB M2: possible but tight. 24GB M2: comfortable.

**Expected runtime:** ~90 minutes on M2 (32 layers × 500 prompts)

---
## Cell 1 — System Check

In [1]:
# ── Install all required packages ────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'datasets', 'huggingface_hub',
                'psutil', 'scipy', 'transformers', '-q'], check=True)

print('All packages installed ✓')

All packages installed ✓


In [2]:
# Fix 1 — install ipywidgets
import subprocess
subprocess.run(['pip', 'install', 'ipywidgets', '-q'], check=True)

# Fix 2 — login using token string directly (no widget needed)
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN")   # paste your token between the quotes
print('Logged in ✓')

HTTPError: Invalid user token. The token stored is invalid. Please run `hf auth login` to update it.

In [ ]:
import torch
import numpy as np
import psutil
import os
import warnings
warnings.filterwarnings('ignore')

# ── Device ────────────────────────────────────────────────────────
DEVICE = 'mps'  if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available()          else 'cpu'

# ── Memory ────────────────────────────────────────────────────────
ram_gb = psutil.virtual_memory().total / 1e9
ram_available_gb = psutil.virtual_memory().available / 1e9

print('=' * 55)
print('  SYSTEM CHECK')
print('=' * 55)
print(f'  Device:           {DEVICE}')
print(f'  Total RAM:        {ram_gb:.1f} GB')
print(f'  Available RAM:    {ram_available_gb:.1f} GB')
print()

# Llama 3 8B in bf16 needs ~16GB
if ram_gb >= 24:
    print('  ✓ 24GB+ — comfortable. Llama 3 8B in bf16 fits easily.')
    MODEL_DTYPE = torch.bfloat16
elif ram_gb >= 16:
    print('  ⚠ 16GB — tight. Close everything else before proceeding.')
    print('  ⚠ Using bf16. Do not run other apps during this notebook.')
    MODEL_DTYPE = torch.bfloat16
else:
    print('  ✗ Less than 16GB — Llama 3 8B will not fit.')
    print('  → Use Llama 3.2 3B instead: change MODEL_NAME below.')
    MODEL_DTYPE = torch.bfloat16

print('=' * 55)

# ── Config ────────────────────────────────────────────────────────
# If you have < 16GB, change to 'meta-llama/Llama-3.2-3B'
MODEL_NAME = 'microsoft/phi-2'
N_CALIBRATION = 500    # GSM8K calibration prompts
N_VALIDATION  = 100    # held-out prompts
MAX_LEN       = 128    # longer than GPT-2 — reasoning prompts are longer
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'\n  Model:       {MODEL_NAME}')
print(f'  Calibration: {N_CALIBRATION} GSM8K prompts')
print(f'  Validation:  {N_VALIDATION} held-out prompts')
print(f'  Max length:  {MAX_LEN} tokens')

  SYSTEM CHECK
  Device:           mps
  Total RAM:        17.2 GB
  Available RAM:    4.9 GB

  ⚠ 16GB — tight. Close everything else before proceeding.
  ⚠ Using bf16. Do not run other apps during this notebook.

  Model:       microsoft/phi-2
  Calibration: 500 GSM8K prompts
  Validation:  100 held-out prompts
  Max length:  128 tokens


---
## Cell 2 — Load GSM8K Dataset

GSM8K is grade school math reasoning.
Each problem has a question and a step-by-step answer ending with `#### number`.
You use the question as prompt and the final numerical answer as completion.

In [ ]:
# pip install datasets if not installed
from datasets import load_dataset

print('Loading GSM8K...')
gsm8k = load_dataset('gsm8k', 'main')

def extract_answer(answer_text):
    """
    GSM8K answers end with '#### 42' where 42 is the final number.
    Extract the final number as the completion.
    """
    parts = answer_text.split('####')
    if len(parts) > 1:
        return ' ' + parts[-1].strip()
    return ' ' + answer_text.strip()[:30]

def build_pairs(split, n):
    """
    Build (prompt, completion) pairs from GSM8K split.
    Prompt = question text.
    Completion = the final numerical answer.
    """
    pairs = []
    for item in split.select(range(n)):
        prompt     = item['question'].strip()
        completion = extract_answer(item['answer'])
        pairs.append((prompt, completion))
    return pairs

# Use test split for calibration (500) and train split for validation (100)
# They are different so there is no data leakage
CALIBRATION = build_pairs(gsm8k['test'],  N_CALIBRATION)
VALIDATION  = build_pairs(gsm8k['train'], N_VALIDATION)

print(f'Calibration: {len(CALIBRATION)} pairs')
print(f'Validation:  {len(VALIDATION)} pairs')
print(f'\nExample prompt:     {CALIBRATION[0][0][:80]}...')
print(f'Example completion: {CALIBRATION[0][1]}')

Loading GSM8K...
Calibration: 500 pairs
Validation:  100 pairs

Example prompt:     Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
Example completion:  18


In [ ]:
from huggingface_hub import model_info
try:
    info = model_info('meta-llama/Llama-3.2-70B')
    print('Access confirmed ✓')
    print(f'Model: {info.modelId}')
except Exception as e:
    print(f'Access issue: {e}')
    print('Request access at: huggingface.co/meta-llama/Llama-3.2-3B')

Access issue: 404 Client Error. (Request ID: Root=1-69e077f8-30b61d504600c01f48e7d9ca;0e33448e-6561-49d5-99d3-4bfa6bac9364)

Repository Not Found for url: https://huggingface.co/api/models/meta-llama/Llama-3.2-70B.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Request access at: huggingface.co/meta-llama/Llama-3.2-3B


In [ ]:
import psutil
available = psutil.virtual_memory().available / 1e9
total     = psutil.virtual_memory().total / 1e9
print(f'Total:     {total:.1f} GB')
print(f'Available: {available:.1f} GB')

if available >= 12:
    print('✓ Safe to load Llama 3 8B — proceed to Cell 3')
elif available >= 8:
    print('⚠ Marginal — use Llama 3.2 3B instead')
    print('  Change MODEL_NAME to: meta-llama/Llama-3.2-3B')
else:
    print('✗ Not enough — close more applications first')

Total:     17.2 GB
Available: 6.5 GB
✗ Not enough — close more applications first


---
## Cell 3 — Load Llama 3 8B Model

Loading in bf16 halves the memory requirement vs fp32.
This takes 2-3 minutes on M2.

In [ ]:
import psutil
avail = psutil.virtual_memory().available / 1e9
print(f'Available: {avail:.1f} GB')
print('✓ Safe' if avail >= 10 else '✗ Close more apps first')

Available: 6.5 GB
✗ Close more apps first


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
# no padding_side override needed for Llama 3.2 1B

model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype       = MODEL_DTYPE,
    device_map        = DEVICE,
    low_cpu_mem_usage = True,
)
model_fp.eval()

N_LAYERS   = model_fp.config.num_hidden_layers
HIDDEN_DIM = model_fp.config.hidden_size
VOCAB_SIZE = model_fp.config.vocab_size
N_PARAMS   = sum(p.numel() for p in model_fp.parameters())

print(f'\n  Layers:     {N_LAYERS}')
print(f'  Hidden dim: {HIDDEN_DIM}')
print(f'  Vocab:      {VOCAB_SIZE:,}')
print(f'  Parameters: {N_PARAMS:,}')
print(f'  Dtype:      {MODEL_DTYPE}')
print(f'\nModel loaded successfully.')

Loading microsoft/phi-2...


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


  Layers:     32
  Hidden dim: 2560
  Vocab:      51,200
  Parameters: 2,779,683,840
  Dtype:      torch.bfloat16

Model loaded successfully.


---
## Cell 4 — Tokenize Dataset

In [ ]:
def tokenize_pairs(data, max_len=MAX_LEN):
    """
    Same as Stage 2 — tokenize both full text and prompt only.
    full_enc   = used for KL-D and activations
    prompt_enc = used to find where completion starts (for accuracy)
    """
    full_enc = tokenizer(
        [p + c for p, c in data],
        return_tensors='pt', padding=True,
        truncation=True, max_length=max_len
    ).to(DEVICE)

    prompt_enc = tokenizer(
        [p for p, c in data],
        return_tensors='pt', padding=True,
        truncation=True, max_length=max_len
    ).to(DEVICE)

    return full_enc, prompt_enc

print('Tokenizing calibration set...')
cal_enc, cal_prompt_enc = tokenize_pairs(CALIBRATION)

print('Tokenizing validation set...')
val_enc, val_prompt_enc = tokenize_pairs(VALIDATION)

print(f'Calibration: {cal_enc["input_ids"].shape}')
print(f'Validation:  {val_enc["input_ids"].shape}')

Tokenizing calibration set...
Tokenizing validation set...
Calibration: torch.Size([500, 128])
Validation:  torch.Size([100, 107])


---
## Cell 5 — All Utility Functions

**Critical difference from GPT-2:**

The `get_layer_linears` function handles the different layer structure.
Llama 3 has `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`.
GPT-2 had `c_attn, c_proj, c_fc, c_proj`.

**Critical efficiency change:**

`kl_divergence_inplace` stores FP32 outputs once and reuses them.
This avoids copying the 8B model 32 times — which would need 1TB of memory.

In [ ]:
# ── Get linear layers for a specific transformer block ────────────
def get_layer_linears(model, layer_idx):
    """
    Returns all linear layers in transformer block layer_idx.

    Llama 3 block structure:
      self_attn.q_proj  ← Query projection
      self_attn.k_proj  ← Key projection
      self_attn.v_proj  ← Value projection
      self_attn.o_proj  ← Output projection
      mlp.gate_proj     ← MLP gate (SwiGLU activation)
      mlp.up_proj       ← MLP up projection
      mlp.down_proj     ← MLP down projection

    Returns list of (name, layer) tuples.
    """
    block = model.model.layers[layer_idx]
    return [
        ('q_proj',    block.self_attn.q_proj),
        ('k_proj',    block.self_attn.k_proj),
        ('v_proj',    block.self_attn.v_proj),
        ('o_proj',    block.self_attn.o_proj),
        ('gate_proj', block.mlp.gate_proj),
        ('up_proj',   block.mlp.up_proj),
        ('down_proj', block.mlp.down_proj),
    ]


# ── INT8 quantization ─────────────────────────────────────────────
def quantize_int8_inplace(layer):
    """
    Symmetric per-tensor INT8 quantization in-place.
    Modifies layer.weight.data directly.
    Save originals before calling. Restore after measuring.
    """
    with torch.no_grad():
        W = layer.weight.data.float()
        s = W.abs().max() / 127.0 + 1e-8
        layer.weight.data = (
            (W / s).round().clamp(-127, 127) * s
        ).to(layer.weight.dtype)


# ── INT4 quantization ─────────────────────────────────────────────
def quantize_int4_inplace(layer):
    """
    Symmetric per-tensor INT4 quantization in-place.
    16 discrete levels — more aggressive than INT8.
    """
    with torch.no_grad():
        W = layer.weight.data.float()
        s = W.abs().max() / 7.0 + 1e-8
        layer.weight.data = (
            (W / s).round().clamp(-7, 7) * s
        ).to(layer.weight.dtype)


# ── Store FP32 outputs once ────────────────────────────────────────
def store_fp32_outputs(model, enc, batch_size=4):
    """
    Run the full-precision model once on all prompts.
    Store the logits on CPU.

    This is the key efficiency improvement over Stage 2.
    Instead of copying the entire model 32 times,
    we run FP32 once and store the outputs.

    batch_size: how many prompts to process at once.
    Lower = less memory. Higher = faster.
    Start with 4, reduce to 1 if you get OOM.

    Returns:
        list of tensors, each shape [1, seq_len, vocab_size] on CPU
    """
    print('  Storing FP32 outputs (runs once, reused for all 32 layers)...')
    fp32_logits = []
    n = enc['input_ids'].shape[0]

    with torch.no_grad():
        for i in range(n):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)
            logits = model(input_ids=ids, attention_mask=mask).logits
            fp32_logits.append(logits.cpu())

            if (i + 1) % 50 == 0:
                print(f'    {i+1}/{n} prompts processed')

    print(f'  FP32 outputs stored: {len(fp32_logits)} tensors')
    return fp32_logits


# ── KL-Divergence using stored FP32 outputs ───────────────────────
def kl_from_stored(model_q, enc, fp32_logits):
    """
    Compute KL-Divergence between stored FP32 outputs
    and the current (quantized) model's outputs.

    No deepcopy needed.
    No second model in memory.
    Just the one model + the stored logit tensors.
    """
    kl_vals = []
    n = enc['input_ids'].shape[0]

    with torch.no_grad():
        for i in range(n):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)

            lq = model_q(input_ids=ids, attention_mask=mask).logits
            lp = fp32_logits[i].to(DEVICE)

            p  = torch.softmax(lp.float(), dim=-1).clamp(1e-9, 1.0)
            q  = torch.softmax(lq.float(), dim=-1).clamp(1e-9, 1.0)
            kl = (p * (p.log() - q.log())).sum(dim=-1).mean()
            kl_vals.append(kl.item())

    return float(np.mean(kl_vals))


# ── Token accuracy ────────────────────────────────────────────────
def token_accuracy(model, enc, prompt_enc):
    correct = total = 0
    with torch.no_grad():
        n = enc['input_ids'].shape[0]
        for i in range(n):
            ids    = enc['input_ids'][i].unsqueeze(0)
            mask   = enc['attention_mask'][i].unsqueeze(0)
            plen   = int(prompt_enc['attention_mask'][i].sum().item())
            flen   = int(mask.sum().item())
            logits = model(input_ids=ids, attention_mask=mask).logits
            for t in range(plen, flen - 1):
                correct += int(logits[0,t].argmax().item() == ids[0,t+1].item())
                total   += 1
    return (correct / total * 100.0) if total > 0 else 0.0


# ── Mutual information estimator ──────────────────────────────────
def mutual_information(X, Y, n_bins=15):
    """
    Histogram-based MI estimator.
    I(X;Y) = H(X) + H(Y) - H(X,Y)

    n_bins=15 for n=500 prompts:
    500 / 15^2 = 2.2 samples per bin — acceptable
    500 / 10^2 = 5.0 samples per bin — better but fewer bins
    n_bins=10 is safer if results look noisy
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    X = (X - X.min()) / (X.max() - X.min() + 1e-9)
    Y = (Y - Y.min()) / (Y.max() - Y.min() + 1e-9)
    joint, _, _ = np.histogram2d(X, Y, bins=n_bins)
    joint = joint / (joint.sum() + 1e-9)
    px, py = joint.sum(axis=1), joint.sum(axis=0)
    def H(p):
        p = p[p > 1e-9]
        return float(-np.sum(p * np.log2(p)))
    return max(H(px) + H(py) - H(joint.flatten()), 0.0)


print('All utility functions defined.')
print('Key difference from GPT-2: in-place approach — no deepcopy of 8B model.')

All utility functions defined.
Key difference from GPT-2: in-place approach — no deepcopy of 8B model.


---
## Cell 6 — EXP 1: Ground Truth KL-D Map

**This is the most time-consuming cell — about 45 minutes on M2.**

For each of 32 layers:
1. Save that layer's 7 weight matrices (~200MB each layer)
2. Quantize to INT4 in-place
3. Run 500 prompts, compute KL-D against stored FP32 outputs
4. Restore original weights

Memory at peak: 1 model (16GB) + stored FP32 logits (~2GB) + 7 weight matrices saved (~200MB)

In [ ]:
print('=' * 55)
print('EXP 1 — Ground Truth KL-D Map')
print('=' * 55)

# Step 1: Store FP32 outputs ONCE
# This runs the model once on all 500 prompts and saves the logits
# Then we reuse these stored logits for all 32 layer experiments
print('\nStep 1: Store FP32 calibration outputs...')
cal_fp32_logits = store_fp32_outputs(model_fp, cal_enc)

# Step 2: Y_task for MI estimation in EXP 1 Option B
# Y = average top-token probability from FP32 model
Y_task_cal = np.array([
    cal_fp32_logits[i].float().softmax(dim=-1).max(dim=-1).values.mean().item()
    for i in range(len(cal_fp32_logits))
])

# Step 3: Layer-wise KL-D map
print('\nStep 2: Layer-wise KL-D map (32 layers)...')
kl_gt = []

for layer_idx in range(N_LAYERS):
    # Get all linear layers in this block
    named_layers = get_layer_linears(model_fp, layer_idx)

    # Save original weights (only 7 matrices, not the whole model)
    originals = {name: layer.weight.data.clone()
                 for name, layer in named_layers}

    # Quantize this layer to INT4 in-place
    for name, layer in named_layers:
        quantize_int4_inplace(layer)

    # Measure KL-D against stored FP32 outputs
    kl = kl_from_stored(model_fp, cal_enc, cal_fp32_logits)
    kl_gt.append(kl)

    # Restore original weights immediately
    for name, layer in named_layers:
        layer.weight.data = originals[name]

    print(f'  Layer {layer_idx:2d}: KL-D = {kl:.6f}')

kl_gt = np.array(kl_gt)
print(f'\nMost degraded layer:  {np.argmax(kl_gt)}  (KL = {kl_gt.max():.5f})')
print(f'Least degraded layer: {np.argmin(kl_gt)}  (KL = {kl_gt.min():.5f})')
print(f'Ratio: {kl_gt.max()/kl_gt.min():.1f}×')

EXP 1 — Ground Truth KL-D Map

Step 1: Store FP32 calibration outputs...
  Storing FP32 outputs (runs once, reused for all 32 layers)...
    50/500 prompts processed
    100/500 prompts processed
    150/500 prompts processed
    200/500 prompts processed
    250/500 prompts processed
    300/500 prompts processed
    350/500 prompts processed


---
## Cell 7 — Option A and B Scores (T-Selection)

Same logic as GPT-2 Stage 2 EXP 1.
Expected on Llama 3 8B with 500 prompts:
- Option B Spearman ρ should be significantly positive (not negative like GPT-2)
- 500 prompts give the MI estimator enough density to work reliably

In [ ]:
from scipy import stats

print('Computing Option A and B importance scores...')

# ── Option A: Weight Frobenius Norm ───────────────────────────────
# Theoretically invalid (weights are input-independent, I(T;X)=0)
# But we measure it for the comparison
option_a = []
for layer_idx in range(N_LAYERS):
    named_layers = get_layer_linears(model_fp, layer_idx)
    total_norm = sum(
        torch.norm(layer.weight.data.float(), 'fro').item()
        for _, layer in named_layers
    )
    option_a.append(total_norm)
option_a = np.array(option_a)
print(f'Option A computed: range {option_a.min():.1f} to {option_a.max():.1f}')

# ── Option B: Activation MI I(Q(h_l); Y) ─────────────────────────
# Theoretically valid (activations are input-dependent)
# Using MLP output activations (same as GPT-2 experiments)
print('\nComputing Option B: I(Q(h_l); Y) per layer...')
option_b = []

def get_mlp_activations(model, enc, layer_idx, n_prompts=100):
    """
    Extract MLP output activations at layer_idx.
    Uses a subset of prompts (default 100) for speed.
    Full 500 would be slow; 100 gives enough density for MI estimation.

    For Llama 3: MLP output = after down_proj
    Hook placed on the full MLP module output.
    """
    acts = []
    hook = model.model.layers[layer_idx].mlp.register_forward_hook(
        lambda m, i, o: acts.append(o.detach().float().cpu())
    )
    with torch.no_grad():
        for i in range(min(n_prompts, enc['input_ids'].shape[0])):
            ids  = enc['input_ids'][i].unsqueeze(0)
            mask = enc['attention_mask'][i].unsqueeze(0)
            model(input_ids=ids, attention_mask=mask)
    hook.remove()
    return torch.cat(acts, dim=0)


# Use first 100 prompts for activation MI (faster)
Y_task_100 = Y_task_cal[:100]

for layer_idx in range(N_LAYERS):
    # Full precision activations
    acts = get_mlp_activations(model_fp, cal_enc, layer_idx, n_prompts=100)

    # Simulate INT4 quantization on activations
    scale  = acts.abs().max() / 7.0 + 1e-8
    acts_q = (acts / scale).round().clamp(-7, 7) * scale

    # Project to scalar per prompt using random projection
    # Johnson-Lindenstrauss: reduces 4096-dim to 64-dim
    # This preserves pairwise distances approximately
    # and makes MI estimation tractable
    proj     = torch.randn(HIDDEN_DIM, 64) / np.sqrt(64)
    acts_proj = (acts_q @ proj).norm(dim=-1).mean(dim=-1).numpy()

    mi = mutual_information(acts_proj, Y_task_100)
    option_b.append(mi)

    if layer_idx % 8 == 0:
        print(f'  Layer {layer_idx:2d}: I(Q(h);Y) = {mi:.4f} bits')

option_b = np.array(option_b)

# ── Spearman Correlation ──────────────────────────────────────────
rho_a, p_a = stats.spearmanr(option_a, kl_gt)
rho_b, p_b = stats.spearmanr(option_b, kl_gt)

print(f'\nSpearman Rank Correlation with Ground Truth KL-D:')
print(f'  Option A (Weight Norm):    ρ = {rho_a:.4f}  (p = {p_a:.4f})')
print(f'  Option B (Activation MI):  ρ = {rho_b:.4f}  (p = {p_b:.4f})')
print(f'\n  → {'Option B wins ✓' if rho_b > rho_a else 'Option A wins — investigate'}')
print(f'  → On Llama 3 8B with 500 prompts, Option B ρ should be positive and significant')

---
## Cell 8 — EXP 4: Layer Information Plane

In [ ]:
print('EXP 4 — Layer Information Plane...')
print('(Using 100 prompts for speed)')

# Input entropy per prompt (X signal)
def input_entropy(enc, n=100):
    H = []
    for i in range(min(n, enc['input_ids'].shape[0])):
        ids = enc['input_ids'][i].cpu().numpy()
        _, counts = np.unique(ids, return_counts=True)
        p = counts / counts.sum()
        H.append(float(-np.sum(p * np.log2(p + 1e-9))))
    return np.array(H)

X_input = input_entropy(cal_enc, n=100)

# Build fully INT4 quantized model for EXP 4 comparison
# We quantize all layers at once, then measure
print('Quantizing entire model to INT4 for displacement measurement...')
all_originals = {}
for layer_idx in range(N_LAYERS):
    named_layers = get_layer_linears(model_fp, layer_idx)
    all_originals[layer_idx] = {
        name: layer.weight.data.clone()
        for name, layer in named_layers
    }
    for name, layer in named_layers:
        quantize_int4_inplace(layer)

print('Running EXP 4 per layer...')
ix_fp, iy_fp = [], []
ix_q4, iy_q4 = [], []

proj = torch.randn(HIDDEN_DIM, 64) / np.sqrt(64)   # fixed projection

for layer_idx in range(N_LAYERS):
    # First restore to FP32 temporarily to get FP32 activations
    for name, layer in get_layer_linears(model_fp, layer_idx):
        layer.weight.data = all_originals[layer_idx][name]

    af = get_mlp_activations(model_fp, cal_enc, layer_idx, n_prompts=100)
    Xf = (af @ proj).norm(dim=-1).mean(dim=-1).numpy()
    ix_fp.append(mutual_information(Xf, X_input))
    iy_fp.append(mutual_information(Xf, Y_task_cal[:100]))

    # Re-quantize to INT4
    for name, layer in get_layer_linears(model_fp, layer_idx):
        quantize_int4_inplace(layer)

    aq = get_mlp_activations(model_fp, cal_enc, layer_idx, n_prompts=100)
    Xq = (aq @ proj).norm(dim=-1).mean(dim=-1).numpy()
    ix_q4.append(mutual_information(Xq, X_input))
    iy_q4.append(mutual_information(Xq, Y_task_cal[:100]))

    if layer_idx % 8 == 0:
        print(f'  Layer {layer_idx:2d}: FP[{ix_fp[-1]:.3f},{iy_fp[-1]:.3f}]  '
              f'INT4[{ix_q4[-1]:.3f},{iy_q4[-1]:.3f}]')

# Restore all weights to FP32
for layer_idx in range(N_LAYERS):
    for name, layer in get_layer_linears(model_fp, layer_idx):
        layer.weight.data = all_originals[layer_idx][name]
print('All weights restored to FP32.')

ix_fp = np.array(ix_fp);  iy_fp = np.array(iy_fp)
ix_q4 = np.array(ix_q4);  iy_q4 = np.array(iy_q4)
arrow_len = np.sqrt((ix_q4 - ix_fp)**2 + (iy_q4 - iy_fp)**2)

print(f'\nMost displaced:  Layer {np.argmax(arrow_len)}  '
      f'(arrow = {arrow_len.max():.4f})')
print(f'Least displaced: Layer {np.argmin(arrow_len)}  '
      f'(arrow = {arrow_len.min():.4f})')
print(f'Ratio: {arrow_len.max()/arrow_len.min():.1f}×')

---
## Cell 9 — Compute β and Bit Allocation

In [ ]:
def compute_beta(kl_gt, arrow_len, alpha=0.5):
    def norm(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)
    beta_raw = alpha * norm(kl_gt) + (1 - alpha) * norm(arrow_len)
    return beta_raw / (beta_raw.mean() + 1e-9)

def compute_bit_allocation(beta, target_bits, n_layers):
    sorted_layers = np.argsort(beta)[::-1]
    k = int(round(n_layers * (target_bits - 4.0) / 4.0))
    k = max(0, min(k, n_layers))
    alloc = {int(l): ('int8' if r < k else 'int4')
             for r, l in enumerate(sorted_layers)}
    actual = np.mean([8 if v=='int8' else 4 for v in alloc.values()])
    return alloc, actual

# Run α sensitivity at 6-bit to find best α for Llama 3 8B
print('α sensitivity at 6-bit...')
print(f'{"α":>5}  {"INT8 layers (top 16 by β)"}')
print('─' * 55)

best_alpha = 0.5   # will update below after seeing results
for alpha_val in [0.0, 0.2, 0.5, 0.8, 1.0]:
    b = compute_beta(kl_gt, arrow_len, alpha=alpha_val)
    alloc, _ = compute_bit_allocation(b, 6.0, N_LAYERS)
    int8 = sorted([l for l,v in alloc.items() if v=='int8'])
    print(f'{alpha_val:>5.1f}  {int8}')

# Use α = 0.5 as default (update based on above output)
alpha = 0.5
beta  = compute_beta(kl_gt, arrow_len, alpha=alpha)

print(f'\nUsing α = {alpha}')
print(f'β range: {beta.min():.3f} to {beta.max():.3f}')

# Build allocations at multiple bit targets
all_allocations = {}
for target in [4.0, 5.0, 6.0, 7.0, 8.0]:
    alloc, actual = compute_bit_allocation(beta, target, N_LAYERS)
    all_allocations[target] = alloc
    k = sum(1 for v in alloc.values() if v=='int8')
    print(f'  {target:.0f}-bit target: {k} layers INT8, {N_LAYERS-k} layers INT4  '
          f'(actual avg = {actual:.1f} bits)')

---
## Cell 10 — Run All Experiments and Measure KL-D

Applies each quantization policy in-place, measures KL-D, restores weights.
No deepcopy of 8B model needed at any point.

In [ ]:
def apply_allocation_inplace(model, allocation):
    """
    Apply a bit allocation dict to the model in-place.
    Returns saved originals so you can restore afterwards.
    """
    saved = {}
    for layer_idx, prec in allocation.items():
        named_layers = get_layer_linears(model, layer_idx)
        saved[layer_idx] = {
            name: layer.weight.data.clone()
            for name, layer in named_layers
        }
        for name, layer in named_layers:
            if   prec == 'int8': quantize_int8_inplace(layer)
            elif prec == 'int4': quantize_int4_inplace(layer)
    return saved

def restore_weights(model, saved):
    """Restore model weights from saved dict."""
    for layer_idx, name_weights in saved.items():
        named_layers = dict(get_layer_linears(model, layer_idx))
        for name, weights in name_weights.items():
            named_layers[name].weight.data = weights

# ── Store validation FP32 outputs ─────────────────────────────────
print('Storing FP32 validation outputs...')
val_fp32_logits = store_fp32_outputs(model_fp, val_enc)

# ── Run all configurations ────────────────────────────────────────
print('\nRunning all quantization experiments...')
print('=' * 60)

results = []

def evaluate(label, allocation, bits, config_type):
    print(f'\n  [{config_type}] {label}  ({bits:.1f}-bit)')

    if allocation:
        saved = apply_allocation_inplace(model_fp, allocation)

    kl_cal = kl_from_stored(model_fp, cal_enc, cal_fp32_logits)
    kl_val = kl_from_stored(model_fp, val_enc, val_fp32_logits)

    print(f'    KL(cal)={kl_cal:.5f}   KL(val)={kl_val:.5f}')

    if allocation:
        restore_weights(model_fp, saved)

    results.append({
        'label': label, 'bits': bits,
        'kl_cal': kl_cal, 'kl_val': kl_val,
        'type': config_type
    })

# FP32 baseline — no quantization
evaluate('FP32 Baseline', {}, 32.0, 'baseline')

# Uniform baselines
evaluate('Uniform INT8',
         {i:'int8' for i in range(N_LAYERS)}, 8.0, 'uniform')
evaluate('Uniform INT4',
         {i:'int4' for i in range(N_LAYERS)}, 4.0, 'uniform')

# Stage 3 IB allocations
for target, alloc in sorted(all_allocations.items()):
    actual = np.mean([8 if v=='int8' else 4 for v in alloc.values()])
    evaluate(f'IB-{target:.0f}bit (combined β)', alloc, actual, 'ib')

print('\n' + '=' * 60)
print('All experiments complete.')

---
## Cell 11 — Final Results Summary

In [ ]:
import json

print('=' * 70)
print('LLAMA 3 8B — STAGE 3 RESULTS')
print('=' * 70)
print(f'{"Configuration":<35} {"Bits":>6} {"KL(cal)":>10} {"KL(val)":>10}')
print('─' * 65)
for r in results:
    print(f'{r["label"]:<35} {r["bits"]:>6.1f} '
          f'{r["kl_cal"]:>10.5f} {r["kl_val"]:>10.5f}')

# Pareto check
print(f'\nPareto check — IB vs Uniform at each bit rate:')
print(f'{"Config":<25} {"Bits":>5} {"KL(cal)":>10} {"vs uniform":>15}')
print('─' * 60)
uniform_kl = {8.0: next(r['kl_cal'] for r in results if 'INT8' in r['label']),
               4.0: next(r['kl_cal'] for r in results if 'INT4' in r['label'])}

for r in results:
    if r['type'] == 'ib':
        closest_uniform_kl = uniform_kl.get(
            8.0 if r['bits'] > 6 else 4.0, None
        )
        if closest_uniform_kl:
            reduction = (closest_uniform_kl - r['kl_cal']) / closest_uniform_kl * 100
            better = f'{reduction:+.1f}% vs uniform'
        else:
            better = 'new operating point'
        print(f'{r["label"]:<25} {r["bits"]:>5.1f} {r["kl_cal"]:>10.5f} {better:>15}')

# T-selection result
print(f'\nT-Selection:')
print(f'  Option A ρ = {rho_a:.4f}  (Weight Norm — theoretically invalid)')
print(f'  Option B ρ = {rho_b:.4f}  (Activation MI — T = Q(h))')
print(f'  → {'Option B confirmed ✓' if rho_b > rho_a else 'investigate further'}')

# Save
output = {
    'model': MODEL_NAME, 'n_layers': N_LAYERS,
    'n_calibration': N_CALIBRATION, 'alpha': alpha,
    'kl_gt': kl_gt.tolist(), 'arrow_len': arrow_len.tolist(),
    'beta': beta.tolist(),
    'rho_a': rho_a, 'rho_b': rho_b,
    'results': results
}
with open('llama3_stage3_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nResults saved to llama3_stage3_results.json')